# Intent-SIM Pipeline (9B)

Full pipeline from `combined_samples.json`:
- **Intent simulation T=0.7** (Qwen recommended 1.0, paper uses 0.5)
- **Oracle Q&A via local Qwen 3.5 9B** (paper uses GPT-3.5)
- **Few-shot clarification examples** in the answer prompt

## Step 0: Setup & Config

In [ ]:
import json
import re
import string
import requests
import os
from datetime import datetime
from tqdm.auto import tqdm

# === CONFIGURATION ===
LLAMA_SERVER_URL = "http://localhost:8080/v1/chat/completions"

INPUT_FILE = "../output/eval/oracle_9b.json"
OUTPUT_DIR = "../output/eval/9b"

## Step 1: Load Data

In [3]:
with open(INPUT_FILE) as f:
    samples = json.load(f)

amb = [s for s in samples if s["is_ambiguous"]]
unamb = [s for s in samples if not s["is_ambiguous"]]
print(f"Loaded {len(samples)} samples: {len(amb)} ambiguous, {len(unamb)} unambiguous")

Loaded 400 samples: 200 ambiguous, 200 unambiguous


In [ ]:

for sample in samples:
    # We only care about modifying the ambiguous samples as per your logic
    if not sample["is_ambiguous"]:
        continue
    
    # Remove the specified keys
    # Using .pop(key, None) ensures the code doesn't crash if a key is already missing
    sample.pop("oracle_clarifying_question", None)
    sample.pop("oracle_clarifying_responses", None)

# Save the modified samples to the new file
with open("/home/pulse/Desktop/intent-sim-clean/output/eval/just_combined2.json", 'w', encoding='utf-8') as f:
    json.dump(samples, f, indent=4)

print(f"Processed samples and saved to {"OUTPUT_FILE"}")


## Step 2: Generate Clarifying Questions (llama-server, T=0.0)

In [ ]:
import copy
def generate_clarifying_questions(dataset, server_url=LLAMA_SERVER_URL):
    new_dataset = []


    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = f"{OUTPUT_DIR}/9b_clarifying_log_{timestamp}.jsonl"

    with open(log_file, "w", encoding="utf-8") as log_f:
        for entry in tqdm(dataset, desc="Generating Clarifying Questions"):
            try:
                new_entry = copy.deepcopy(entry)
                question = new_entry["input_x"]

                messages = [
                    {
                        "role": "system",
                        "content": (
                            "You are a professional question analyzer designed to disambiguate possible interpretations of asked questions.\n"
                            "Your task is to respond with a follow up question to an input question.\n"
                            "Ask a clarification to separate an intent, your question should be consise and relevant to the question they are asking.\n"
                            "Do not question the premise or accuracy of the question — assume it is correct as stated.\n"
                            "You should give your output in the format Follow-Up Question: <>\n"
                            "Below are some examples:\n\n"
                            "Example 1:\n"
                            "Question: Who won the US open?\n"
                            "Follow-Up Question: Are you referring to Men's Singles or Women's Singles?\n\n"
                            "Example 2:\n"
                            "Question: How many Grammy Awards does Whitney Houston have?\n"
                            "Follow-Up Question: Are you referring to the number of Grammy Awards Whitney Houston won, or the number of Grammy Awards Whitney Houston was nominated for?\n\n"
                            "Example 3:\n"
                            "Question: Tell me about the Apollo program.\n"
                            "Follow-Up Question: Are you referring to the overall history, a specific mission like Apollo 11, or the technology?\n"
                        )
                    },
                    {"role": "user", "content": f"Question: {question}"}
                ]

                response = client.chat.completions.create(
                    model="local-model",
                    messages=messages,
                    temperature=0.0,
                    max_tokens=80,
                    stop=["\n", "Follow-Up Answer", "Answer"]
                )
                raw_output = response.choices[0].message.content.strip()

                # Logging
                log_entry = {
                    "input_x": question,
                    "raw_output": raw_output,
                    "greedy_clarifying_question": raw_output
                }
                log_f.write(json.dumps(log_entry) + "\n")
                log_f.flush()

                new_entry["intent_sim"] = {"greedy_clarifying_question": raw_output}
                new_dataset.append(new_entry)
            except Exception as e:
                print(f"Error for '{question}': {e}")
    return new_dataset

In [12]:
samples[0]

{'id': '-7729753255484758871',
 'input_x': 'Who did america fight during world war 1?',
 'is_ambiguous': False,
 'gold_output_y_star': ['Germany', 'the Austro - Hungarian Empire']}

In [13]:
intent_samples = generate_clarifying_questions(samples)
print(f"Sample: {intent_samples[0]['input_x']}")
print(f"  -> {intent_samples[0]['intent_sim']}")

Generating Clarifying Questions: 100%|██████████| 400/400 [19:24<00:00,  2.91s/it]

Sample: Who did america fight during world war 1?
  -> {'greedy_clarifying_question': 'Follow-Up Question: Are you asking about the primary Allied powers America fought alongside against Germany and Austria-Hungary, or are you inquiring about specific conflicts or theaters of operation within the war?'}


In [ ]:
with open("../output/eval/9b/9b_intent_sim_questions.json", 'w', encoding='utf-8') as f:
    json.dump(intent_samples, f, indent=4)

## Step 3: Simulate Intents (llama-server)

In [ ]:
def sample_intents(input_x, clarifying_question, server_url=LLAMA_SERVER_URL):
    system_content = (
        "You are a simulated user responding to a clarifying question.\n"
        "Your task is to select EXACTLY ONE option from the choices presented in the follow-up question.\n"
        "Respond to the follow-up question by picking EXACTLY ONE of the choices from the question.\n"
        "IMPORTANT: Provide ONLY the specific interpretation. Do not use conversational filler, "
        "do not write complete sentences, and do not describe your choice. Be as brief as possible.\n"
        "Here are some examples below:\n\n"
        "Example 1:\n"
        "Question: Who won the US open?\n"
        "Follow-Up Question: Are you referring to Men's Singles or Women's Singles?\n"
        "Answer: Women's Singles\n\n"
        "Example 2:\n"
        "Question: How many Grammy Awards does Whitney Houston have?\n"
        "Follow-Up Question: Are you referring to the number of Grammy Awards Whitney Houston won, or the number of Grammy Awards Whitney Houston was nominated for?\n"
        "Answer: The number of Grammy Awards Whitney Houston won\n\n"
        "Example 3:\n"
        "Question: Tell me about apple\n"
        "Follow-Up Question: Are you referring to the tech company Apple, or the fruit apple?\n"
        "Answer: The tech company Apple"
        "Example 4:\n"
        "Question: What is the matrix?\n"
        "Follow-Up Question: Are you referring to the movie, the mathematical concept, or the organizational structure?\n"
        "Answer: the mathematical concept"
    )


    payload = {
        "model": "local-model",
        "messages": [
            {"role": "system", "content": system_content},
            {"role": "user", "content": f"Question: {input_x}\n{clarifying_question}"}
        ],
        "temperature": 0.7,
        "max_tokens": 30
    }

    collected_responses = []
    batch_sizes = [4, 4, 2]

    for batch_n in batch_sizes:
        payload["n"] = batch_n
        response = requests.post(server_url, json=payload)
        response.raise_for_status()
        data = response.json()

        assert len(data["choices"]) == batch_n, "incorrect batching"
        for choice in data["choices"]:
            collected_responses.append(choice["message"]["content"].strip())

    return collected_responses

In [30]:
sample_intents("When was mercury discovered?", "Follow-Up Question: Are you referring to the planet or the chemical element?")

['The planet Mercury.',
 'The planet Mercury.',
 'The planet.',
 'The planet Mercury.',
 'The planet Mercury.',
 'The chemical element.',
 'The chemical element.',
 'The chemical element.',
 'The planet Mercury.',
 'The chemical element.']

In [35]:
sample_intents("Tell me about apple.", "Follow-Up Question: Are you referring to company Apple, or the fruit apple?")

['The fruit apple.',
 'The fruit apple.',
 'The tech company Apple.',
 'The company Apple.',
 'The tech company Apple.',
 'Company Apple.',
 'Company Apple.',
 'The fruit apple.',
 'Company Apple.',
 'Company Apple.']

In [41]:
# Phase 1: Generate all intents
for entry in tqdm(intent_samples, desc="Step 1/2: Generating Intents"):
    input_x = entry["input_x"]
    clarifying_question = entry["intent_sim"]["greedy_clarifying_question"]
    sampled_intents = sample_intents(input_x, clarifying_question)

    entry["intent_sim"] = {
        "greedy_clarifying_question": clarifying_question,
        "generated_intents": sampled_intents,
        "uncertainty_score": None,
        "clusters": {},
        "cluster_probabilities": {}
    }

print(f"Generated intents for {len(intent_samples)} samples")
print(f"Sample intents: {intent_samples[0]['intent_sim']['generated_intents'][:3]}")

Step 1/2: Generating Intents: 100%|██████████| 400/400 [31:08<00:00,  4.67s/it]

Generated intents for 400 samples
Sample intents: ['The primary Allied powers America fought alongside against Germany and Austria-Hungary.', 'The primary Allied powers America fought alongside against Germany and Austria-Hungary.', 'The primary Allied powers America fought alongside against Germany and Austria-Hungary.']


In [48]:
intent_samples[6]

{'id': '7556791920106701864',
 'input_x': "What is l's real name from death note?",
 'is_ambiguous': True,
 'all_interpretations': [{'disambiguated_question': "What is L's real name from Death Note according to the guidebook Death Note 13: How to Read?",
   'answer': ['L Lawliet']},
  {'disambiguated_question': "What is L's real name from Death Note according to the American film of Death Note?",
   'answer': ['Lebens Dorn']},
  {'disambiguated_question': 'What is the real name of the actor who voices L in Japanese, anime?',
   'answer': ['Kappei Yamaguchi']},
  {'disambiguated_question': 'What is the real name of the actor who voices L in Japanese?',
   'answer': ['Shin-ichiro Miki']},
  {'disambiguated_question': 'What is the real name of the actor who voices L in English?',
   'answer': ['Alessandro Juliani']},
  {'disambiguated_question': 'What is the real name of the actor who portrays L in the films?',
   'answer': ['Kenichi Matsuyama']},
  {'disambiguated_question': 'What is the

In [ ]:
with open("../output/eval/9b/9b_intent_sim_simulations.json", 'w', encoding='utf-8') as f:
    json.dump(intent_samples, f, indent=4)

## Step 4: NLI Clustering & Entropy (DeBERTa-MNLI)

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from scipy.stats import entropy

MODEL_NAME = "microsoft/deberta-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, use_safetensors=False)
nli_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nli_model.to(device)
print(f"DeBERTa-MNLI loaded on {device}")


def check_equivalence(q, a1, a2):
    text1 = f"{a1}"
    text2 = f"{a2}"

    inputs_left = tokenizer(text1, text2, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        logits_left = nli_model(**inputs_left).logits
    pred_left = torch.argmax(logits_left, dim=1).item()

    inputs_right = tokenizer(text2, text1, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        logits_right = nli_model(**inputs_right).logits
    pred_right = torch.argmax(logits_right, dim=1).item()

    ENTAILMENT_LABEL = 2
    return (pred_left == ENTAILMENT_LABEL) or (pred_right == ENTAILMENT_LABEL)


def cluster_and_calculate_entropy(clarifying_question, simulated_responses):
    S = len(simulated_responses)
    G = {i: set() for i in range(S)}

    for i in range(S - 1):
        for j in range(i + 1, S):
            if check_equivalence(clarifying_question, simulated_responses[i], simulated_responses[j]):
                G[i].add(j)
                G[j].add(i)

    visited = set()
    clusters = []
    for i in range(S):
        if i not in visited:
            cluster = set()
            stack = [i]
            while stack:
                node = stack.pop()
                if node not in visited:
                    visited.add(node)
                    cluster.add(node)
                    stack.extend(G[node] - visited)
            clusters.append(cluster)

    cluster_probs = [len(c) / S for c in clusters]
    u = entropy(cluster_probs, base=2)
    return u, clusters, cluster_probs

## BATCHED VERSION

In [ ]:
import torch
from scipy.stats import entropy

ENTAILMENT_LABEL = 2
BATCH_SIZE = 32 # increase this to 64 or 128 if your GPU has a lot of VRAM

def cluster_and_calculate_entropy(clarifying_question, simulated_responses):
    S = len(simulated_responses)
    G = {i: set() for i in range(S)}

    # 1. Prepare all pairs for batching
    pairs = []
    indices = []
    for i in range(S - 1):
        for j in range(i + 1, S):
            # Currently ignoring the question as per your commented out code, 
            # just comparing the answers.
            text1 = f"{simulated_responses[i]}"
            text2 = f"{simulated_responses[j]}"
            
            # Add left-to-right and right-to-left pairs
            pairs.append((text1, text2))
            pairs.append((text2, text1))
            indices.append((i, j))

    # 2. Run batched inference
    preds = []
    for batch_start in range(0, len(pairs), BATCH_SIZE):
        batch_pairs = pairs[batch_start : batch_start + BATCH_SIZE]
        texts1 = [p[0] for p in batch_pairs]
        texts2 = [p[1] for p in batch_pairs]
        
        # Tokenize the whole batch at once with padding
        inputs = tokenizer(texts1, texts2, return_tensors="pt", padding=True, truncation=True).to(device)
        
        with torch.no_grad():
            logits = nli_model(**inputs).logits
            
        # Get predictions and move back to CPU
        batch_preds = torch.argmax(logits, dim=1).cpu().tolist()
        preds.extend(batch_preds)

    # 3. Reconstruct the graph from batched predictions
    for k, (i, j) in enumerate(indices):
        pred_left = preds[k * 2]
        pred_right = preds[k * 2 + 1]
        
        # If either direction entails, they are equivalent
        if (pred_left == ENTAILMENT_LABEL) or (pred_right == ENTAILMENT_LABEL):
            G[i].add(j)
            G[j].add(i)

    # 4. Standard DFS Clustering
    visited = set()
    clusters = []
    for i in range(S):
        if i not in visited:
            cluster = set()
            stack = [i]
            while stack:
                node = stack.pop()
                if node not in visited:
                    visited.add(node)
                    cluster.add(node)
                    stack.extend(G[node] - visited)
            clusters.append(cluster)

    # Calculate entropy
    cluster_probs = [len(c) / S for c in clusters]
    u = entropy(cluster_probs, base=2)
    
    return u, clusters, cluster_probs

In [ ]:
test_q = "Are you referring to the planet or the chemical element?"
test_q = "Follow-Up Question: Are you referring to company Apple, or the fruit apple?"

test_responses = [
    'The chemical element Mercury.',
    'Mercury element.',
    'The chemical element Mercury.',
    'The chemical element Mercury.',
    'the chemical element.',
    'Mercury is a planet.',
    'The planet Mercury.',
    'The chemical element.',
    'the chemical element.',
    'the Planet Mercury.'
]

test_responses = [
    'The tech company Apple',
    'The tech company Apple',
    'The tech company',
    'The tech company',
    'The company Apple',
    'The company Apple',
    'The fruit',
    'fruit',
    'the apple fruit',
    'the one you eat',
]


# Run the clustering and entropy calculation
u_score, clusters, probs = cluster_and_calculate_entropy(test_q, test_responses)

# Print the results in a readable format
print(f"Results for: '{test_q}'")
print(f"{'='*50}")
print(f"Uncertainty Score (Entropy): {u_score:.4f} bits")
print(f"Number of Clusters: {len(clusters)}")

for idx, cluster_indices in enumerate(clusters):
    print(f"\nCluster {idx+1} (Prob: {probs[idx]:.2f}):")
    # Show the unique responses in this cluster for brevity
    unique_contents = set(test_responses[i] for i in cluster_indices)
    for text in unique_contents:
        print(f"  - {text}")

print(f"{'='*50}")

Results for: 'Follow-Up Question: Are you referring to company Apple, or the fruit apple?'
Uncertainty Score (Entropy): 1.2955 bits
Number of Clusters: 3

Cluster 1 (Prob: 0.60):
  - The company Apple
  - The tech company Apple
  - The tech company

Cluster 2 (Prob: 0.30):
  - The fruit
  - fruit
  - the apple fruit

Cluster 3 (Prob: 0.10):
  - the one you eat


In [54]:
intent_samples[0]["intent_sim"]["greedy_clarifying_question"],intent_samples[0]["intent_sim"]["generated_intents"]

('Follow-Up Question: Are you asking about the primary Allied powers America fought alongside against Germany and Austria-Hungary, or are you inquiring about specific conflicts or theaters of operation within the war?',
 ['The primary Allied powers America fought alongside against Germany and Austria-Hungary.',
  'The primary Allied powers America fought alongside against Germany and Austria-Hungary.',
  'The primary Allied powers America fought alongside against Germany and Austria-Hungary.',
  'The primary Allied powers America fought alongside against Germany and Austria-Hungary.',
  'The primary Allied powers America fought alongside against Germany and Austria-Hungary.',
  'The primary Allied powers America fought alongside.',
  'The primary Allied powers America fought alongside against Germany and Austria-Hungary.',
  'The primary Allied powers America fought alongside against Germany and Austria-Hungary.',
  'The primary Allied powers America fought alongside against Germany an

In [55]:
# Phase 2: Cluster and compute entropy
for sample in tqdm(intent_samples, desc="Step 2/2: Calculating Uncertainty"):
    clarifying_question = sample["intent_sim"]["greedy_clarifying_question"]
    sampled_intents = sample["intent_sim"]["generated_intents"]

    if not sampled_intents:
        print(f"FAILURE: no intents for {sample['id']}")
        break

    score, clusters, probs = cluster_and_calculate_entropy(clarifying_question, sampled_intents)

    sample["intent_sim"].update({
        "uncertainty_score": score,
        "clusters": [sorted(list(c)) for c in clusters],
        "cluster_probabilities": probs
    })

print("Done. Clustering complete.")

Step 2/2: Calculating Uncertainty: 100%|██████████| 400/400 [01:52<00:00,  3.57it/s]

Done. Clustering complete.


### Free Memory (Delete and unload the NLI model)

In [ ]:
import torch
import gc

print(f"Before Memory Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
print(f"Before Memory Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

if 'nli_model' in globals():
    nli_model.cpu()

del nli_model
del tokenizer 
gc.collect()
torch.cuda.empty_cache()

print(f"Memory Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
print(f"Memory Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

Before Memory Allocated: 1558.72 MB
Before Memory Reserved: 2030.00 MB
Memory Allocated: 9.12 MB
Memory Reserved: 22.00 MB


## Step 5: Sort by Uncertainty & Save Intermediate

In [2]:
# Sort descending by uncertainty score
intent_samples.sort(key=lambda x: x["intent_sim"]["uncertainty_score"], reverse=True)

# Quick stats
amb = [s for s in intent_samples if s["is_ambiguous"]]
unamb = [s for s in intent_samples if not s["is_ambiguous"]]

amb_scores = [s["intent_sim"]["uncertainty_score"] for s in amb]
unamb_scores = [s["intent_sim"]["uncertainty_score"] for s in unamb]
amb_clusters = [len(s["intent_sim"]["clusters"]) for s in amb]
unamb_clusters = [len(s["intent_sim"]["clusters"]) for s in unamb]

print(f"Ambiguous   — mean entropy: {sum(amb_scores)/len(amb_scores):.3f}, mean clusters: {sum(amb_clusters)/len(amb_clusters):.1f}")
print(f"Unambiguous — mean entropy: {sum(unamb_scores)/len(unamb_scores):.3f}, mean clusters: {sum(unamb_clusters)/len(unamb_clusters):.1f}")

from collections import Counter
print(f"\nAmbiguous cluster distribution:   {sorted(Counter(amb_clusters).items())}")
print(f"Unambiguous cluster distribution: {sorted(Counter(unamb_clusters).items())}")

# Save scored samples
scored_path = f"../output/eval/9b/9b_scored_samples.json"
with open(scored_path, "w") as f:
    json.dump(intent_samples, f, indent=2)
print(f"\nSaved scored samples to {scored_path}")


Ambiguous   — mean entropy: 0.191, mean clusters: 1.3
Unambiguous — mean entropy: 0.114, mean clusters: 1.2

Ambiguous cluster distribution:   [(1, 147), (2, 52), (3, 1)]
Unambiguous cluster distribution: [(1, 166), (2, 33), (4, 1)]

Saved scored samples to ../output/eval/9b/9b_scored_samples.json


## Step 7 & 8: Generate Direct & Clarified Answers (llama-server)

In [ ]:
# System prompt for direct QA (same as original)
DIRECT_SYSTEM_PROMPT = """You are a simple question answering model, your only task is to respond with a definite answer in one short word, phrase, date, name etc.
Below are some examples:

Question 1: When did the simpsons first air on television?
Answer 1: December 17, 1989

Question 2: What is the sink that looks like a toilet?
Answer 2: Bidet

Question 3: How many hours are there in a day?
Answer 3: 24

Question 4: When did the us first go into the middle east?
Answer 4: 1833

Question 5: What is the tallest building in the world?
Answer 5: Burj Khalifa"""

# System prompt for clarified QA, includes few-shot examples WITH clarification
CLARIFIED_SYSTEM_PROMPT = """You are a simple question answering model, your only task is to respond with a definite answer in one short word, phrase, date, name etc.
Sometimes you will receive a follow-up question and answer that clarifies the user's intent. Use this information to give a more precise answer.
Below are some examples:

Question 1: Who won the US open?
Follow-Up Question: Are you referring to Men's Singles or Women's Singles?
Follow-Up Answer: Women's Singles.
Answer 1: Coco Gauff

Question 2: How many Grammy Awards does Whitney Houston have?
Follow-Up Question: Are you referring to the number of Grammy Awards Whitney Houston won, or the number of Grammy Awards Whitney Houston was nominated for?
Follow-Up Answer: The number of Grammy Awards Whitney Houston won.
Answer 2: 6

Question 3: When did the simpsons first air on television?
Answer 3: December 17, 1989

Question 4: What is the tallest building in the world?
Answer 4: Burj Khalifa"""


def find_gold_interpretation(oracle_responses, gold_output_y_star):
    """Find the oracle_clarifying_response whose answers match gold_output_y_star."""
    gold_normalized = {normalize_text(a) for a in gold_output_y_star}
    for resp in oracle_responses:
        resp_normalized = {normalize_text(a) for a in resp["answer"]}
        if any(g in r or r in g for g in gold_normalized for r in resp_normalized):
            return resp
    return None


def ask_llm_direct(user_content, server_url):
    payload = {
        "messages": [
            {"role": "system", "content": DIRECT_SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ],
        "temperature": 0.0,
        "top_p": 1.0,
        "max_tokens": 30,
        "stop": ["\n", "Question:", "Answer:"]
    }
    response = requests.post(server_url, json=payload)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"].strip()


def ask_llm_clarified(user_content, server_url):
    """Answer with the clarification-aware system prompt."""
    payload = {
        "messages": [
            {"role": "system", "content": CLARIFIED_SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ],
        "temperature": 0.0,
        "top_p": 1.0,
        "max_tokens": 30,
        "stop": ["\n", "Question:", "Answer:"]
    }
    response = requests.post(server_url, json=payload)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"].strip()


def generate_evaluation_responses(intent_samples, server_url=LLAMA_SERVER_URL):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = f"../output/logs/9b/reworked_9b_eval_log_{timestamp}.jsonl"

    with open(log_file, "w", encoding="utf-8") as log_f:
        for entry in tqdm(intent_samples, desc="Generating Evaluation Responses"):
            input_x = entry["input_x"]

            # Direct answer (all questions)
            direct_answer = ask_llm_direct(f"Question: {input_x}", server_url)
            entry["evaluation_responses"] = {"direct_answer": direct_answer}

            # Clarified answer (ambiguous only)
            if entry["is_ambiguous"]:
                gold_interp = find_gold_interpretation(
                    entry["oracle_clarifying_responses"],
                    entry["gold_output_y_star"]
                )

                if gold_interp and gold_interp["clarification_response"]:
                    clarified_prompt = (
                        f"Question: {input_x}\n"
                        f"Follow-Up Question: {entry['oracle_clarifying_question']}\n"
                        f"Follow-Up Answer: {gold_interp['clarification_response']}\n"
                        f"Answer:"
                    )
                    clarified_answer = ask_llm_clarified(clarified_prompt, server_url)
                    entry["evaluation_responses"]["clarified_answer"] = clarified_answer
                    entry["evaluation_responses"]["clarified_interpretation"] = gold_interp
                else:
                    print(f"ERROR, COULDNT FIND gold interp for {entry['id']}")
                    entry["evaluation_responses"]["clarified_answer"] = None
                    entry["evaluation_responses"]["clarified_interpretation"] = None

            # Log
            log_entry = {
                "id": entry["id"],
                "input_x": input_x,
                "is_ambiguous": entry["is_ambiguous"],
                "evaluation_responses": entry["evaluation_responses"]
            }
            log_f.write(json.dumps(log_entry) + "\n")
            log_f.flush()

In [ ]:
def normalize_text(s):
    """Lower text, remove articles, punctuation, and extra whitespace."""
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = s.translate(str.maketrans('', '', string.punctuation))
    s = ' '.join(s.split())
    return s

generate_evaluation_responses(intent_samples)
print(f"Done. Generated responses for {len(intent_samples)} samples")

# Save
eval_path = f"../output/eval/9b/9b_evaluation_responses.json"
with open(eval_path, "w") as f:
    json.dump(intent_samples, f, indent=2)
print(f"Saved to {eval_path}")

## Step 9: Evaluate

In [63]:
def is_correct(predicted, gold_answers):
    """Check if predicted answer matches any gold answer (bidirectional substring match)."""
    if predicted is None:
        return False
    pred_norm = normalize_text(predicted)
    for gold in gold_answers:
        gold_norm = normalize_text(gold)
        if pred_norm in gold_norm or gold_norm in pred_norm:
            return True
    return False


# Compute per-entry correctness
for entry in intent_samples:
    gold = entry["gold_output_y_star"]
    direct = entry["evaluation_responses"]["direct_answer"]
    entry["direct_correct"] = is_correct(direct, gold)

    if entry["is_ambiguous"] and "clarified_answer" in entry["evaluation_responses"]:
        clarified = entry["evaluation_responses"]["clarified_answer"]
        entry["clarified_correct"] = is_correct(clarified, gold) if clarified else False
    else:
        entry["clarified_correct"] = None

amb = [s for s in intent_samples if s["is_ambiguous"]]
unamb = [s for s in intent_samples if not s["is_ambiguous"]]
print(f"Ambiguous:   {sum(s['direct_correct'] for s in amb)}/{len(amb)} direct correct, "
      f"{sum(s['clarified_correct'] for s in amb)}/{len(amb)} clarified correct")
print(f"Unambiguous: {sum(s['direct_correct'] for s in unamb)}/{len(unamb)} direct correct")

Ambiguous:   39/200 direct correct, 55/200 clarified correct
Unambiguous: 56/200 direct correct


In [64]:
# Budget evaluation
# Data is already sorted by uncertainty_score descending.

def accuracy_at_budget(samples, budget_pct):
    k = int(len(samples) * budget_pct / 100)
    correct = 0
    for i, entry in enumerate(samples):
        if entry["is_ambiguous"] and i < k:
            correct += entry["clarified_correct"]
        else:
            correct += entry["direct_correct"]
    return correct / len(samples)


budgets = [0, 10, 20, 30, 100]
print("\n--- Accuracy at Each Budget ---")
print(f"{'Budget':<10} {'Accuracy':<12} {'Correct':<10} {'Total':<8}")
print("-" * 40)

for b in budgets:
    acc = accuracy_at_budget(intent_samples, b)
    correct = int(acc * len(intent_samples))
    print(f"b={b:>3}%     {acc:.4f}       {correct:<10} {len(intent_samples)}")

direct_acc = accuracy_at_budget(intent_samples, 0)
print(f"\nBaseline (direct-only): {direct_acc:.4f}")
for b in [10, 20, 30]:
    acc = accuracy_at_budget(intent_samples, b)
    gain = acc - direct_acc
    pct_gain = (gain / direct_acc * 100) if direct_acc > 0 else 0
    print(f"b={b}%: {acc:.4f} (gain: {gain:+.4f}, {pct_gain:+.1f}%)")


--- Accuracy at Each Budget ---
Budget     Accuracy     Correct    Total   
----------------------------------------
b=  0%     0.2375       95         400
b= 10%     0.2550       102        400
b= 20%     0.2600       104        400
b= 30%     0.2600       104        400
b=100%     0.2775       111        400

Baseline (direct-only): 0.2375
b=10%: 0.2550 (gain: +0.0175, +7.4%)
b=20%: 0.2600 (gain: +0.0225, +9.5%)
b=30%: 0.2600 (gain: +0.0225, +9.5%)


In [65]:
# AUROC
from scipy.stats import rankdata

def compute_auroc(samples):
    labels = []
    scores = []
    for entry in samples:
        labels.append(1 if entry["is_ambiguous"] else 0)
        scores.append(entry["intent_sim"]["uncertainty_score"])

    n_pos = sum(labels)
    n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return float('nan')

    ranks = rankdata(scores)
    pos_rank_sum = sum(r for r, l in zip(ranks, labels) if l == 1)
    auroc = (pos_rank_sum - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return auroc


auroc = compute_auroc(intent_samples)
print(f"\n--- AUROC ---")
print(f"AUROC: {auroc:.4f}")
print(f"(Random baseline: 0.5000)")


--- AUROC ---
AUROC: 0.5514
(Random baseline: 0.5000)


In [4]:
# Summary
print("\n" + "=" * 60)
print("EVALUATION SUMMARY — Intent-SIM (9B Model)")
print("=" * 60)
print(f"{'Metric':<30} {'Value':<15}")
print("-" * 45)
print(f"{'AUROC':<30} {auroc:.4f}")
print(f"{'Direct Accuracy (b=0%)':<30} {accuracy_at_budget(intent_samples, 0):.4f}")
for b in [10, 20, 30]:
    acc = accuracy_at_budget(intent_samples, b)
    gain = acc - direct_acc
    print(f"{'Accuracy (b=' + str(b) + '%)':<30} {acc:.4f} ({gain:+.4f})")
print(f"{'Clarify-All Accuracy (b=100%)':<30} {accuracy_at_budget(intent_samples, 100):.4f}")
print("-" * 45)
print(f"{'Ambiguous Direct Correct':<30} {sum(s['direct_correct'] for s in amb)}/{len(amb)}")
print(f"{'Ambiguous Clarified Correct':<30} {sum(s['clarified_correct'] for s in amb)}/{len(amb)}")
print(f"{'Unambiguous Direct Correct':<30} {sum(s['direct_correct'] for s in unamb)}/{len(unamb)}")
print("=" * 60)

EVALUATION SUMMARY — Intent-SIM (9B Model)
Metric                         Value          
---------------------------------------------
AUROC                          0.5514
Direct Accuracy (b=0%)         0.2375
Accuracy (b=10%)               0.2550 (+0.0175)
Accuracy (b=20%)               0.2600 (+0.0225)
Accuracy (b=30%)               0.2600 (+0.0225)
Clarify-All Accuracy (b=100%)  0.2775
---------------------------------------------
Ambiguous Direct Correct       39/200
Ambiguous Clarified Correct    55/200
Unambiguous Direct Correct     56/200
